In [0]:
-- -- Data Cube: Comprehensive sales analytics view combining fact and dimension tables
-- -- This cube provides all measures and dimensions needed for KPI creation
-- -- Stored as a permanent Delta table in the Gold layer

-- CREATE OR REPLACE TABLE electronics_retailer_clg.gold.sales_data_cube AS
-- SELECT 
--   -- Fact Keys & Transaction Details
--   f.order_number,
--   f.order_date,
--   f.delivery_date,
--   f.delivery_time_days,
  
--   -- Time Dimensions
--   YEAR(f.order_date) AS order_year,
--   QUARTER(f.order_date) AS order_quarter,
--   MONTH(f.order_date) AS order_month,
--   DAYOFWEEK(f.order_date) AS order_day_of_week,
--   DATE_FORMAT(f.order_date, 'EEEE') AS order_day_name,
  
--   -- Customer Dimensions
--   f.customerkey,
--   COALESCE(c.gender, f.gender) AS customer_gender,
--   COALESCE(c.continent, f.continent) AS customer_continent,
  
--   -- Product Dimensions
--   f.productkey,
--   COALESCE(p.category, f.product_category) AS product_category,
--   p.unit_price_usd AS product_list_price,
  
--   -- Store Dimensions
--   f.storekey,
--   COALESCE(s.Country, f.store_country) AS store_country,
--   s.State AS store_state,
--   s.Square_Meters AS store_size_sqm,
--   s.Open_Date AS store_open_date,
  
--   -- Channel Dimension
--   f.channel AS sales_channel,
  
--   -- Currency Information
--   f.currency_code,
--   f.exchange_rate,
  
--   -- Measures (Facts/Metrics)
--   f.unit_price_usd AS transaction_unit_price,
--   f.quantity AS units_sold,
--   f.revenue_usd AS revenue_usd,
  
--   -- Calculated Measures for KPIs
--   f.revenue_usd / NULLIF(f.quantity, 0) AS average_unit_revenue,
--   CASE 
--     WHEN f.delivery_time_days <= 3 THEN 'Fast' 
--     WHEN f.delivery_time_days <= 7 THEN 'Standard'
--     ELSE 'Slow' 
--   END AS delivery_speed_category
  
-- FROM electronics_retailer_clg.gold.fact_sales f

-- -- Left join with dimension tables to enrich with additional attributes
-- LEFT JOIN electronics_retailer_clg.gold.dim_customers c
--   ON f.customerkey = c.customer_key
  
-- LEFT JOIN electronics_retailer_clg.gold.dim_products p
--   ON f.productkey = p.productkey
  
-- LEFT JOIN electronics_retailer_clg.default.store s
--   ON f.storekey = s.StoreKey;

In [0]:

%sql
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_kpis.curated_sales AS
SELECT
    f.order_number,
    f.line_item,

    f.order_date,
    f.year,
    f.month,

    f.customerkey,
    c.gender,
    c.continent,

    f.storekey,
    COALESCE(s.store_country, 'Online') AS country,

    f.productkey,
    p.category,
    p.subcategory,

    f.quantity,

    f.unit_price_usd,
    f.exchange,

    f.revenue_usd,
    f.delivery_days,

    f.channel

FROM electronics_retailer_clg.gold.fact_sales f
LEFT JOIN electronics_retailer_clg.gold.dim_customers c
    ON f.customerkey = c.customerkey
LEFT JOIN electronics_retailer_clg.gold.dim_products p
    ON f.productkey = p.productkey
LEFT JOIN electronics_retailer_clg.gold.dim_stores s
    ON f.storekey = s.storekey;